# Group 6 - Geometry Optimization (Armin & Lukas)

This jupyter notebook is a snippet of the total codebase we used to investigate geometry optimization using quantum computing methods. The total codebase can be found on the respective GitHub repository:

[VU_Quantum_Computing-Github](https://github.com/LukasMeinschad/vu_quantum_computing)

> This notebook focuses on the bond scan of the hydrogen molecule H2 using a VQE approach and is designed to be simple to use on the AQT QC.

## Module Import

The following packages are needed to run the code:

+ qiskit 
+ PySCF 
+ numpy
+ qiskit-aer
+ qiskit-nature

In [ ]:
import numpy as np

from qiskit_aer import AerSimulator
from qiskit_algorithms.optimizers import COBYLA
from qiskit_algorithms import MinimumEigensolverResult
from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_nature.second_q.transformers import FreezeCoreTransformer, ActiveSpaceTransformer 
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

## Configuration 

The following cell is the central configuration to set the number of shots and configure the Backend to be used. For testing reasons we used the AerSimulator.

**Calculation Setup**

We want to run a bond-scan for the hydrogen molecule in the range from 0.54Å to 0.94Å with 5 points. At each point we build a UCCSD Ansatz with 1 reps (see step 3 for details).

+ Each VQE evaluation will be run with a total number of **20** optimization steps (MAXITER) using COBYLA as a classical optimizer
+ The number of SHOTS per expectation value is set to **70**
+ There is a total of **14** non-identity Pauli terms in the qubit Hamiltonian for H2 in this basis
+ As seen in the last step of this notebook one VQE minimization for a specific bond length will use 20 * 70 * 14 = 19600 shots
+ This multiplies to a total shot count of **98,000** for the whole bond scan with **5** points

**NOTE**: We already limited the total shot count for the bond scan severely.
          However, if it is still too high: the SHOTS and MAXITER parameters can be adjusted further down in the following cell.

In [ ]:
BACKEND = AerSimulator() # Set the AQT backend here
SHOTS = 70 # Set the Number of Shots here
MAXITER = 20 # Set the Maximum Number of VQE Iterations (for this circuit 17 is the lower bound of the optimizer)
OPT_LEVEL = 1 # Set the Optimization Level of the Circuit Compilation

The following cell creates the measurement circuit and estimates our operator based on shots and backend defined in the configuration cell above.

In [ ]:
def build_measurement_circuit_for_pauli(base_circuit, pauli_label: str):
    """
    Measurment circuit for a given Pauli string.
    Basis change for X/Y terms and then measure_all().
    """
    qc = base_circuit.copy()
    n = len(pauli_label)
    if qc.num_qubits != n:
        raise ValueError(f"Pauli label length {n} != circuit qubits {qc.num_qubits}")

    for qubit in range(n):
        op = pauli_label[n - 1 - qubit]  # qubit-0 is on the right
        if op == "I":
            continue
        if op == "X":
            qc.h(qubit)
        elif op == "Y":
            qc.sdg(qubit)
            qc.h(qubit)
        elif op == "Z":
            pass
        else:
            raise ValueError(f"Unsupported Pauli op: {op}")

    qc.measure_all()
    return qc


def counts_to_pauli_expectation(counts: dict[str, int], pauli_label: str) -> float:
    """
    Converts Z-basis counts to <P>; assumes basis change has already been applied.
    """
    n = len(pauli_label)
    active_qubits = [q for q in range(n) if pauli_label[n - 1 - q] != "I"]
    if not active_qubits:
        return 1.0

    shots = int(sum(counts.values()))
    if shots <= 0:
        raise ValueError("No shots in counts")

    exp = 0.0
    for bitstring, c in counts.items():
        parity = 0
        for q in active_qubits:
            parity ^= (bitstring[n - 1 - q] == "1")
        exp += (-1.0 if parity else 1.0) * float(c)

    return exp / float(shots)


def backend_expectation_sparsepauliop(
    *,
    backend,
    base_circuit,
    params: np.ndarray,
    operator,          # SparsePauliOp
    shots: int,
    optimization_level: int | None = None,
    initial_layout=None,
    ):
    """
    Determines the expectation value of `operator` with respect to the state
    prepared by `base_circuit` with `params`, using traditional backend.run().

    Hardware note (AQT, etc.): the measurement circuits are transpiled with a 
    fixed `initial_layout` so the Pauli-string qubit indexing stays consistent 
    with the compiled circuit layout.
    """
    labels = list(operator.paulis.to_labels())
    coeffs = np.asarray(operator.coeffs)

    meas_circuits = []
    keep_idx = []

    energy = 0.0
    for i, (lab, coeff) in enumerate(zip(labels, coeffs)):
        coeff_r = float(np.real(coeff))
        if np.isclose(coeff_r, 0.0):
            continue
        if set(lab) == {"I"}:
            energy += coeff_r
            continue
        # Bind parameters to the base circuit
        bound_circuit = base_circuit.assign_parameters(params)
        meas_circuits.append(build_measurement_circuit_for_pauli(bound_circuit, lab))
        keep_idx.append(i)

    if meas_circuits:
        # Transpile measurement circuits for the backend
        try:
            from qiskit import transpile
            meas_circuits = transpile(
                meas_circuits,
                backend=backend,
                optimization_level=int(optimization_level) if optimization_level is not None else None,
                initial_layout=initial_layout,
            )
        except Exception:
            # If transpile fails for whatever reason
            # fall back to running as-is
            pass

        # Traditional backend.run() execution
        job = backend.run(meas_circuits, shots=int(shots))
        res = job.result()

        for j, i in enumerate(keep_idx):
            counts = res.get_counts(j)
            term_exp = counts_to_pauli_expectation(counts, labels[i])
            energy += float(np.real(coeffs[i])) * float(term_exp)

    return float(energy)

## Step 1: Build the Fermionic Hamiltonian parametrized with the bond length

In [ ]:
def build_h2_fermionic_hamiltonian(
    distance: float,
    *,
    basis: str,
    charge: int,
    spin: int,
    freeze_core: bool,
    active_space: tuple[int, int] | None,  # (num_electrons, num_spatial_orbitals)
):
    """
    Returns the fermionic Hamiltonian which is parametrized on the bond distance x
    """
    driver = PySCFDriver(
        atom=f"H 0 0 0; H 0 0 {float(distance)}",
        unit=DistanceUnit.ANGSTROM,
        basis=basis,
        charge=charge,
        spin=spin,
    )
    problem = driver.run()

    if freeze_core:
        problem = FreezeCoreTransformer().transform(problem)

    if active_space is not None:
        nelec, norb = map(int, active_space)
        problem = ActiveSpaceTransformer(
            num_electrons=nelec, num_spatial_orbitals=norb
        ).transform(problem)

    fermionic_ham = problem.second_q_ops()[0]
    return fermionic_ham, problem

## Step 2: Map the Fermionic Hamiltonian to the Qubit Hamiltonian

In this step, we use the Jordan-Wigner mapping to convert the Fermionic Hamiltonian to qubit Hamiltonian. Further transformations are tested in the original codebase on GitHub.

In [ ]:
def build_h2_qubit_hamiltonian(
    distance: float,
    *,
    basis: str = "sto3g",
    charge: int = 0,
    spin: int = 0,
    freeze_core: bool = False,
    active_space: tuple[int, int] | None = None,
):
    """
    Builds a qubit Hamiltonian for H2 for a given distance using the Jordan-Wigner mapping 
    """
    fermionic_ham, problem = build_h2_fermionic_hamiltonian(
        distance,
        basis=basis,
        charge=charge,
        spin=spin,
        freeze_core=freeze_core,
        active_space=active_space,
    )

    qubit_ham = JordanWignerMapper().map(fermionic_ham)

    return qubit_ham, problem

# Step 3: Prepare the Ansatz Circuit

In the next step, we prepare the ansatz circuit using the UCCSD method with 1 reps. Different Ansatz methods are tested in the original codebase on GitHub.

In [ ]:
def build_h2_uccsd_ansatz(
    problem,
    *,
    reps: int = 1,
):
    """
    Builds a UCCSD ansatz for H2 using Jordan-Wigner mapping and Hartree-Fock initial state.
    """
    mapper = JordanWignerMapper()

    hf = HartreeFock(
        num_spatial_orbitals=problem.num_spatial_orbitals,
        num_particles=problem.num_particles,
        qubit_mapper=mapper,
    )

    return UCCSD(
        num_spatial_orbitals=problem.num_spatial_orbitals,
        num_particles=problem.num_particles,
        qubit_mapper=mapper,
        initial_state=hf,
        reps=reps,
    )


Hq, problem = build_h2_qubit_hamiltonian(0.74)
ansatz = build_h2_uccsd_ansatz(problem, reps=1)

pm = generate_preset_pass_manager(backend=BACKEND, optimization_level=int(OPT_LEVEL))
isa_ansatz = pm.run(ansatz)

print("\nNumber of operations (after transpile):")
print(isa_ansatz.count_ops())
print("Circuit Depth (after transpile):", isa_ansatz.depth())


# Step 4: Set up the VQE algorithm with the COBYLA optimizer

In [ ]:
def interpret_total_energy(exp_val, problem) -> float:
    """
    Convert Estimator expectation value -> total energy (Ha) via problem.interpret.
    """
    ev = np.real(exp_val)
    if isinstance(ev, np.ndarray):
        ev = ev.item() if ev.size == 1 else float(ev.ravel()[0])

    sol = MinimumEigensolverResult()
    sol.eigenvalue = float(ev)
    return float(problem.interpret(sol).total_energies[0])


def run_vqe_single_point_backend(
    *,
    problem,
    qubit_hamiltonian,
    ansatz,
    shots: int,
    backend=BACKEND,              
    optimization_level: int = 1,
    x0: np.ndarray | None = None,
    maxiter: int,
    seed: int = 42,
    verbose: bool = True,
 ):
    """
    VQE Solver using traditional backend.run()
    """
    # Transpile -> removes High-Level Ops like EvolvedOps for Aer/Hardware
    pm = generate_preset_pass_manager(backend=backend, optimization_level=int(optimization_level))
    isa_ansatz = pm.run(ansatz)

    isa_observables = (
        qubit_hamiltonian.apply_layout(isa_ansatz.layout)
        if hasattr(qubit_hamiltonian, "apply_layout")
        else qubit_hamiltonian
    )

    opt = COBYLA(maxiter=maxiter, rhobeg=1.0, tol=1e-6)

    if x0 is None:
        rng = np.random.default_rng(seed)
        x0 = rng.random(isa_ansatz.num_parameters) if isa_ansatz.num_parameters > 0 else np.array([])

    energies: list[float] = []

    def cost(p: np.ndarray) -> float:
        ev = backend_expectation_sparsepauliop(
            backend=backend,
            base_circuit=isa_ansatz,
            params=np.asarray(p, dtype=float),
            operator=isa_observables,
            shots=int(shots),
            optimization_level=int(optimization_level),
            initial_layout=getattr(isa_ansatz, "layout", None),
        )
        e = interpret_total_energy(ev, problem)
        energies.append(e)
        if verbose:
            print(f"Eval {len(energies):03d}: E = {e:.10f} Ha")
        return float(e)

    if isa_ansatz.num_parameters == 0:
        e = cost(x0)
        return {"fun": float(e), "x": np.array(x0, copy=True), "energies": energies}

    res = opt.minimize(fun=cost, x0=np.asarray(x0, dtype=float))
    return {"fun": float(res.fun), "x": np.array(res.x, copy=True), "energies": energies}


## Step 5: Build the Bond Scan function

This function performs the VQE evaluation at each point of the bond length array and collects the associated energies to sketch the potential energy surface of the H2 molecule.

In [ ]:
def bond_scan_h2_vqe_backend(
    *,
    distances=None,
    reps: int = 1,
    maxiter: int = 120,
    seed: int = 42,
    warm_start: bool = True,
    basis: str = "sto3g",
    active_space: tuple[int, int] | None = (2, 2),
    shots: int,
    backend=None,
    optimization_level: int = 1,
):
    """  
    Performs a bond distance scan for H2 using VQE with traditional backend.run().
    At each distance, the VQE is run to find the ground state energy using
    the UCCSD ansatz and Jordan-Wigner mapping.
    """
    if distances is None:
        distances = np.linspace(0.54, 0.94, 5)

    distances = np.asarray(distances, dtype=float)
    energies = []
    trajectories = []
    params = []
    prev_x = None

    for i, d in enumerate(distances):
        Hq, problem = build_h2_qubit_hamiltonian(
            float(d),
            basis=basis,
            charge=0,
            spin=0,
            freeze_core=False,
            active_space=active_space,
        )
        ansatz = build_h2_uccsd_ansatz(problem, reps=reps)

        x0 = prev_x if (warm_start and prev_x is not None and len(prev_x) == ansatz.num_parameters) else None

        out = run_vqe_single_point_backend(
            problem=problem,
            qubit_hamiltonian=Hq,
            ansatz=ansatz,
            shots=shots,
            backend=backend,
            optimization_level=optimization_level,
            x0=x0,
            maxiter=MAXITER,
            seed=seed,
            verbose=True,
        )

        e = float(out["fun"])
        energies.append(e)
        trajectories.append(list(out["energies"]))
        params.append(np.array(out["x"], copy=True))
        prev_x = params[-1]

        print(f"[{i+1:02d}/{len(distances):02d}] d={d:.4f} Å  E={e:.10f} Ha")

    energies = np.asarray(energies, dtype=float)

    return {"distances": distances, "energies": energies, "convergence": trajectories, "optimal_params": params}

# Example result for the bond scan and the associated convergence at each bond length

![H2 bond scan](h2_bond_scan.png)

![H2 vqe convergence](h2_vqe_convergence.png)

# Final Step

In this final step we run the bond scan function from above. We calculated 5 points between 0.54Å and 0.94Å with 1 reps and a maximum of 20 iterations for the optimizer.

In [ ]:
res_scan = bond_scan_h2_vqe_backend(
    distances=np.linspace(0.54, 0.94, 5),
    reps=1,
    maxiter=MAXITER,
    shots=SHOTS,
    backend=BACKEND,
    optimization_level=OPT_LEVEL,
 )

bond_scan_arr = np.column_stack([res_scan["distances"], res_scan["energies"]])
np.savetxt(
    "bond_scan.dat",
    bond_scan_arr,
    fmt="%.10f",
    header="distance_A  energy_Ha",
 )

traj = res_scan["convergence"]
n_points = len(traj)
max_len = max((len(t) for t in traj), default=0)
conv_mat = np.full((max_len, n_points), np.nan, dtype=float)
for j, t in enumerate(traj):
    t = np.asarray(t, dtype=float)
    conv_mat[: len(t), j] = t

np.savetxt(
    "vqe_convergence.dat",
    conv_mat,
    fmt="%.10f",
    header=" ".join([f"d={d:.6f}A" for d in res_scan["distances"]]),
 )

res_scan["distances"], res_scan["energies"]

# Shot Count Helper

This last cell here is a helper cell to calculate the total number of shots for the VQE algorithm. We first evaluate the number of measured Pauli-Terms, multiply this with the shots. Next we run the VQE algorithm for a max number of 20 iterations and get the total shot count for one specific bond length.

In [ ]:
def num_measured_pauli_terms(qubit_hamiltonian, *, tol: float = 1e-12) -> int:
    """
    Helper function to count the number of non-trivial Pauli terms
    in the qubit Hamiltonian that need to be measured.
    """

    labels = list(qubit_hamiltonian.paulis.to_labels())
    coeffs = np.asarray(qubit_hamiltonian.coeffs)
    n = 0
    for lab, c in zip(labels, coeffs):
        if np.isclose(np.real(c), 0.0, atol=tol):
            continue
        if set(lab) == {"I"}:
            continue
        n += 1
    return n

Hq, problem = build_h2_qubit_hamiltonian(0.74)
n_terms = num_measured_pauli_terms(Hq)
shots_per_eval = SHOTS * n_terms
print("Measured Pauli terms:", n_terms)
print("Shots per energy evaluation:", shots_per_eval)

out = run_vqe_single_point_backend(
    problem=problem,
    qubit_hamiltonian=Hq,
    ansatz=build_h2_uccsd_ansatz(problem, reps=1),
    shots=SHOTS,
    backend=BACKEND,
    optimization_level=OPT_LEVEL,
    maxiter=MAXITER,
    verbose=False,
)

n_evals = len(out["energies"])
total_shots_this_distance = shots_per_eval * n_evals
print("Cost evals:", n_evals)
print("Total shots (for one distance):", total_shots_this_distance)